*Universidade Agostinho Neto · Faculdade de Engenharia (FEUAN) · Departamento de Engenharia Informática (DEI) · **Ciência de Dados 2026/27** · Docente: **Eliseu Domingos***

# PL3 — Do ficheiro à tabela: câmbios, agências e o pandas

**Nome:** ...  **Número:** ...

**Objectivo deste notebook:** obter dados de três fontes (uma API, um JSON e um CSV), transformá-los em `DataFrame`, tratar as datas e resumir por grupos, e depois usar as mesmas ferramentas nos dados do Banco Palanca (Aula 3 da apostila).

*Os dados do Banco Palanca, das agências e do histórico de câmbios são fictícios. O câmbio da Tarefa 1 é real.*

## Como trabalhar neste notebook

Como nas PL anteriores, cada passo tem uma célula de código **já escrita** ou uma célula `# TAREFA` para completar, a saída que **deverá obter** e uma pergunta para responder com números.

Ficheiros que têm de estar na mesma pasta do notebook: `cambio_api_2026-10-20.json`, `cambios_usd_2026.json`, `agencias.csv` e `banco_palanca_cartoes.xlsx`.

Se ainda não instalou as bibliotecas desta semana, faça-o uma vez no terminal, com o `.venv` activo: `python -m pip install requests beautifulsoup4`.

## Tarefa 1 — O câmbio de hoje, pedido a uma API

A tesouraria quer comparar o histórico de câmbios com o câmbio **de hoje**. Vamos pedi-lo a uma API pública e gratuita (Aula 3.4.1).

**Passo 1.1** — Execute a célula. Ela pede os câmbios do dólar à API e, se não houver Internet, usa uma resposta guardada no dia 20 de Outubro de 2026.

In [ ]:
import json
import requests

url = "https://open.er-api.com/v6/latest/USD"
try:
    resposta = requests.get(url, timeout=10)
    resposta.raise_for_status()
    dados_api = resposta.json()
    print("Pedido feito agora: código", resposta.status_code)
except requests.exceptions.RequestException as erro:
    print("Sem ligação; usamos a resposta guardada.", erro)
    with open("cambio_api_2026-10-20.json", encoding="utf-8") as f:
        dados_api = json.load(f)

Pedido feito agora: código 200


*Deverá obter* uma de duas linhas: `Pedido feito agora: código 200` ou `Sem ligação; usamos a resposta guardada.`, seguida da descrição do erro. Nos dois casos, `dados_api` fica com a resposta.

**Pergunta 1.1.** Para que serve o `timeout=10`? O que aconteceria sem o `try … except`, se a rede falhasse?

*Resposta:* O `timeout=10` define um limite de 10 segundos para a resposta da API; se não chegar a tempo, a chamada falha de forma controlada em vez de ficar “pendurada” indefinidamente. Sem o `try … except`, a exceção de rede/timeout quebraria o programa e a execução pararia, em vez de usar a resposta guardada localmente.


In [ ]:
# TAREFA: mostre, com dois print, o tipo de dados_api e a lista das suas chaves
print(type(dados_api))
print(list(dados_api.keys()))


<class 'dict'>
['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix', 'time_last_update_utc', 'time_next_update_unix', 'time_next_update_utc', 'time_eol_unix', 'base_code', 'rates']


*Deverá obter:*

```
<class 'dict'>
['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix', 'time_last_update_utc', 'time_next_update_unix', 'time_next_update_utc', 'time_eol_unix', 'base_code', 'rates']
```

É um **dicionário** de Python, como na Aula 2.3.2. As taxas de câmbio estão dentro da chave `rates`, que é outro dicionário, com uma taxa por moeda.

In [ ]:
# TAREFA: mostre a moeda de base (chave "base_code"), a data da última actualização ("time_last_update_utc")
# e o câmbio do kwanza (dentro de "rates", a chave "AOA")
print(dados_api["base_code"])
print(dados_api["time_last_update_utc"])
print(dados_api["rates"]["AOA"])


USD
Fri, 13 Nov 2026 00:02:31 +0000
916.171827


*Deverá obter algo como* (no dia 20 de Outubro de 2026):

```
USD | Tue, 20 Oct 2026 00:02:32 +0000
938.865208
```

O valor muda todos os dias. **Pergunta 1.2.** Um colega executou a mesma célula ontem e obteve outro número. Algum dos dois está errado? O que deve escrever num relatório que use este valor?

*Resposta:* Não há erro em nenhum dos dois: a API devolve o câmbio atual, e esse valor pode mudar ao longo do tempo conforme a taxa de mercado. O que deve constar no relatório é a data/hora exata da consulta e a fonte, por exemplo: “Câmbio do USD em 20/10/2026 às 00:02:32 UTC: 938,87 AOA/USD” ou equivalente, para deixar claro que o valor é do momento em que foi pedido.


In [ ]:
import pandas as pd

data = pd.to_datetime(dados_api["time_last_update_unix"], unit="s").strftime("%Y-%m-%d")
nome = f"cambio_api_{data}.json"
with open(nome, "w", encoding="utf-8") as f:
    json.dump(dados_api, f, indent=1)
print("Guardado em", nome)

*Deverá obter* `Guardado em cambio_api_AAAA-MM-DD.json`, com a data da última actualização da API. Abra o ficheiro no VS Code: é texto, com a mesma estrutura de chaves que vimos acima.

## Tarefa 2 — O histórico da tesouraria: do JSON ao resumo mensal

A tesouraria enviou o câmbio de compra e de venda de todos os dias úteis de Abril a Setembro, num JSON (Aula 3.3.3).

**Passo 2.1** — Leia o ficheiro e veja a sua estrutura.

In [ ]:
with open("cambios_usd_2026.json", encoding="utf-8") as f:
    dados = json.load(f)
print(list(dados))
print(dados["fonte"])
print(len(dados["cotacoes"]), dados["cotacoes"][0])

['moeda', 'base', 'fonte', 'cotacoes']
Banco Palanca, tesouraria (valores ilustrativos)
131 {'data': '2026-04-01', 'compra': 912.0, 'venda': 921.12}


*Deverá obter:*

```
['moeda', 'base', 'fonte', 'cotacoes']
Banco Palanca, tesouraria (valores ilustrativos)
131 {'data': '2026-04-01', 'compra': 912.0, 'venda': 921.12}
```

A tabela está **dentro** da chave `cotacoes`: uma lista de dicionários, um por dia útil, todos com as mesmas três chaves.

In [ ]:
import pandas as pd

# TAREFA: crie o DataFrame cambio a partir de dados["cotacoes"] e mostre os tipos das colunas (.dtypes)
cambio = pd.DataFrame(dados["cotacoes"])
print(cambio.dtypes)

data       object
compra    float64
venda     float64
dtype: object


*Deverá obter:*

```
data          object
compra    float64
venda     float64
dtype: object
```

**Pergunta 2.1.** Os câmbios já são números (`float64`). E a coluna `data`? Porque é que isso é um problema para responder a "qual foi o câmbio médio de Junho"?

*Resposta:* A coluna `data` está como texto (`object`/`str`), não como data. Isso é um problema porque, sem converter para `datetime`, o pandas não sabe que são datas; não consegue ordenar corretamente, filtrar por mês ou fazer cálculos por período. Para responder a “qual foi o câmbio médio de Junho”, primeiro é preciso transformar a data em data real e depois agrupar por mês.

In [ ]:
# TAREFA: converta a coluna data com pd.to_datetime, crie a coluna mes com .dt.month
# e mostre cambio.dtypes outra vez
cambio["data"] = pd.to_datetime(cambio["data"])
cambio["mes"] = cambio["data"].dt.month
print(cambio.dtypes)

data      datetime64[ns]
compra           float64
venda            float64
mes                int32
dtype: object


*Deverá obter:*

```
data      datetime64[us]
compra           float64
venda            float64
mes                int32
dtype: object
```

Agora `data` é `datetime64` e `mes` é um inteiro de 4 (Abril) a 9 (Setembro). Aqui não foi preciso `format`, porque as datas estão no formato ISO (ano-mês-dia), que não tem ambiguidade.

**Passo 2.2** — Resuma o câmbio de **compra** por mês: número de dias, média, mínimo e máximo.

In [ ]:
# TAREFA: use groupby("mes") e agg com nomes: dias (count), media (mean), minimo (min), maximo (max),
# todos sobre a coluna compra; arredonde a 2 casas
resumo_mes = (
    cambio.groupby("mes")
    .agg(
        dias=("compra", "count"),
        media=("compra", "mean"),
        minimo=("compra", "min"),
        maximo=("compra", "max"),
    )
    .round(2)
)
print(resumo_mes)

     dias   media  minimo  maximo
mes                              
4      22  912.60  911.13  913.36
5      21  914.76  912.61  916.42
6      22  921.93  916.60  924.72
7      23  922.71  920.49  924.44
8      21  921.83  920.17  923.80
9      22  923.73  921.38  926.84


*Deverá obter:*

```
     dias   media  minimo  maximo
mes
4      22  912.60  911.13  913.36
5      21  914.76  912.61  916.42
6      22  921.93  916.60  924.72
7      23  922.71  920.49  924.44
8      21  921.83  920.17  923.80
9      22  923.73  921.38  926.84
```

**Pergunta 2.2.** Em quantos kwanzas subiu o câmbio médio de Abril para Setembro? E em percentagem? Em que mês houve a maior diferença entre o mínimo e o máximo?

*Resposta:* O câmbio médio de compra subiu de 912,60 para 923,73 Kz, isto é, 11,13 Kz mais alto, ou cerca de 1,22% em relação a Abril. A maior diferença entre mínimo e máximo foi em Junho: 924,72 − 916,60 = 8,12 Kz.

In [ ]:
# TAREFA: em que dia o câmbio de compra foi mais alto, e qual foi o valor? (dica: idxmax e loc)
linha_max = cambio.loc[cambio["compra"].idxmax()]
print(linha_max["data"].strftime("%Y-%m-%d"), linha_max["compra"])

2026-09-25 926.84


*Deverá obter:*

```
2026-09-25 926.84
```

`idxmax()` devolve a **etiqueta** da linha com o valor máximo (como o `argmax` do NumPy devolvia a posição); `loc` vai buscar essa linha.

In [ ]:
# TAREFA: calcule a diferença média entre o câmbio de venda e o de compra, em Kz, arredondada a 2 casas
margem_media = round((cambio["venda"] - cambio["compra"]).mean(), 2)
print(margem_media)

9.2


*Deverá obter:*

```
np.float64(9.2)
```

É a margem da tesouraria: compra os dólares a um preço e vende-os a outro, cerca de 1 % mais alto.

**Passo 2.3** — Compare o câmbio médio de Setembro com o câmbio da resposta guardada da API (20 de Outubro).

In [ ]:
with open("cambio_api_2026-10-20.json", encoding="utf-8") as f:
    api_20out = json.load(f)
aoa_20out = api_20out["rates"]["AOA"]
setembro = resumo_mes.loc[9, "media"]
print(round(aoa_20out, 2), setembro, round((aoa_20out / setembro - 1) * 100, 2))

938.87 923.73 1.64


*Deverá obter:*

```
938.87 923.73 1.64
```

**Pergunta 2.3.** O dólar "subiu" 1,64 % de Setembro para 20 de Outubro? **Isto faz sentido comparar?** Pense na origem de cada número (veja a chave `fonte` do Passo 2.1 e a Aula 3.2.3).

*Resposta:* Não faz sentido tratar este 1,64 % como uma subida do mesmo indicador. O valor de 923,73 vem do histórico da tesouraria do Banco Palanca, enquanto 938,87 vem da API pública e representa o câmbio atual do mercado. As duas séries têm fontes e definições diferentes: uma é um histórico interno/ilustrativo e a outra é um mercado atual. Só faz sentido comparar valores da mesma origem e com a mesma metodologia.

## Tarefa 3 — As agências: um CSV "à portuguesa"

Um colega enviou a lista das agências em CSV e queixa-se de que o Excel a abre "toda numa coluna" (Aula 3.3.1).

**Passo 3.1** — **Antes de executar**, escreva aqui quantas colunas espera: ...

In [1]:
ag = pd.read_csv("agencias.csv")
ag.shape

NameError: name 'pd' is not defined

*Deverá obter:*

```
(6, 1)
```

Uma só coluna, como no Excel do colega. Abra `agencias.csv` no VS Code e descubra porquê: que carácter separa as colunas? E as casas decimais?

In [ ]:
# TAREFA: leia outra vez o ficheiro com o separador e o símbolo decimal certos, e mostre ag

import pandas as pd

ag = pd.read_csv("agencias.csv", sep=";", decimal=",")

ag

*Deverá obter:*

```
         agencia provincia    abertura  clientes  depositos_mkz
0        Mutamba    Luanda  02/03/2015      8120        1250.50
1       Talatona    Luanda  15/11/2018      6430         980.25
2         Lobito  Benguela  01/07/2016      2915         410.75
3        Lubango     Huíla  04/02/2019      2408         335.00
4    Huambo Sede    Huambo  13/09/2021      2074         290.40
5  Cabinda Porto   Cabinda  08/05/2023      1505         188.90
```

Seis agências e cinco colunas; os depósitos estão em **milhões de kwanzas**.

In [ ]:
# TAREFA: converta a coluna abertura numa data (as datas estão em dia/mês/ano: use format)
# e mostre a lista dos anos de abertura (.dt.year, .tolist())

ag["abertura"] = pd.to_datetime(
    ag["abertura"],
    format="%d/%m/%Y"
)

# Mostrar a lista dos anos de abertura
ag["abertura"].dt.year.tolist()

*Deverá obter:*

```
[2015, 2018, 2016, 2019, 2021, 2023]
```

**Pergunta 3.1.** Se se esquecesse do `format`, o que aconteceria à data "02/03/2015" da Mutamba? E porque é que, com este ficheiro, o pandas daria um erro em vez de trocar as datas em silêncio? (Aula 3.7.1.)

*Resposta:* ...

**Passo 3.2** — Resuma por província.

In [ ]:
# TAREFA: por provincia, conte as agências e some os clientes e os depósitos (agg com nomes:
# agencias, clientes, depositos_mkz); ordene pelos clientes, do maior para o menor

resumo = ag.groupby("provincia").agg(
    agencias=("agencia", "count"),
    clientes=("clientes", "sum"),
    depositos_mkz=("depositos_mkz", "sum")
)

resumo = resumo.sort_values(
    by="clientes",
    ascending=False
)

resumo

*Deverá obter:*

```
           agencias  clientes  depositos_mkz
provincia
Luanda            2     14550        2230.75
Benguela          1      2915         410.75
Huíla             1      2408         335.00
Huambo            1      2074         290.40
Cabinda           1      1505         188.90
```

**Passo 3.3** — Uma coluna calculada: os depósitos médios por cliente, em kwanzas.

In [ ]:
# TAREFA: crie a coluna dep_cliente_kz = depositos_mkz * 1_000_000 / clientes, arredondada às unidades,
# e mostre as colunas agencia, clientes e dep_cliente_kz, ordenadas por dep_cliente_kz (do maior para o menor)


df["dep_cliente_kz"] = (
    df["depositos_mkz"] * 1_000_000 / df["clientes"]
).round(0)

# Mostrar as colunas solicitadas, ordenadas do maior para o menor
df[["agencia", "clientes", "dep_cliente_kz"]].sort_values(
    by="dep_cliente_kz",
    ascending=False
)

*Deverá obter:*

```
         agencia  clientes  dep_cliente_kz
0        Mutamba      8120        154002.0
1       Talatona      6430        152449.0
2         Lobito      2915        140909.0
4    Huambo Sede      2074        140019.0
3        Lubango      2408        139120.0
5  Cabinda Porto      1505        125515.0
```

**Pergunta 3.3.** A Mutamba tem mais clientes **e** mais depósitos por cliente; a Cabinda Porto tem o valor mais baixo. Que perguntas faria à direcção comercial antes de concluir que os clientes de Cabinda "têm menos dinheiro"? (Olhe também para a coluna `abertura`.)

*Resposta:* **Resposta:** Antes de concluir que os clientes de Cabinda têm menos dinheiro, questionaria a direcção comercial sobre o perfil socioeconómico dos clientes, os seus rendimentos, as actividades económicas da região e os factores que influenciam os depósitos em cada agência.

Também analisaria o tempo de funcionamento das agências, observando a coluna `abertura`. A agência Mutamba foi inaugurada em 2015, enquanto a Cabinda Porto foi inaugurada em 2023, o que significa que a Mutamba tem mais tempo de actividade para conquistar clientes e desenvolver a sua carteira de depósitos.

Além disso, procuraria saber se existem diferenças na quantidade e no perfil dos clientes, bem como nos serviços bancários utilizados. Portanto, os dados apresentados não permitem afirmar que os clientes de Cabinda têm menos dinheiro. Apenas demonstram que a agência Cabinda Porto apresenta um depósito médio por cliente inferior ao da Mutamba.

**Passo 3.4** — Grave a tabela num CSV normal (vírgulas e ponto decimal) e leia-a outra vez.

In [ ]:
ag.to_csv("agencias_limpo.csv", index=False)
de_novo = pd.read_csv("agencias_limpo.csv")
de_novo.dtypes

*Deverá obter:*

```
agencia               str
provincia             str
abertura              str
clientes            int64
depositos_mkz     float64
dep_cliente_kz    float64
dtype: object
```

**Pergunta 3.4.** Gravámos `abertura` como data. Que tipo tem depois de lida outra vez? O que nos diz isto sobre o formato CSV? (Aula 3.3.4.)

*Resposta:* Depois de guardar a coluna `abertura` como data e ler novamente o ficheiro CSV, verificamos que o seu tipo voltou a ser `object`, ou seja, texto. Isto acontece porque o formato CSV não preserva os tipos de dados das colunas, armazenando os valores como texto. Por isso, quando voltamos a importar o ficheiro com Pandas, é necessário converter novamente a coluna para o tipo de data utilizando `pd.to_datetime()`.

## Tarefa 4 — O Banco Palanca com o pandas

Na PL2 tirámos colunas para *arrays* NumPy. Agora fazemos o mesmo trabalho sem sair do `DataFrame`, com os nomes das colunas e das contas.

**Passo 4.1** — Leia os dados e crie a máscara das contas válidas (limite maior do que zero, como na PL2).

In [ ]:
df = pd.read_excel("banco_palanca_cartoes.xlsx")
ok = df["LIMITE_CREDITO"] > 0
print(df.shape, ok.sum())

*Deverá obter:*

```
(30000, 26) 29700
```

**Passo 4.2** — `loc` e `iloc`. O gestor de conta pergunta pela conta `927dd27f-1a76`.

In [ ]:
# TAREFA: crie d = df.set_index("ID") e mostre, com loc, o LIMITE_CREDITO, a IDADE, a PROVINCIA,
# o PAG_1 e o INCUMPRIMENTO dessa conta
d = df.set_index("ID")
d.loc["927dd27f-1a76", ["LIMITE_CREDITO","IDADE", "PROVINCIA","PAG_1", "INCUMPRIMENTO"]]
pos = d.index.get_loc("927dd27f-1a76")
colunas = [
    d.columns.get_loc("LIMITE_CREDITO"),
    d.columns.get_loc("IDADE"),
    d.columns.get_loc("PROVINCIA"),
    d.columns.get_loc("PAG_1"),
    d.columns.get_loc("INCUMPRIMENTO")
]

*Deverá obter:*

```
LIMITE_CREDITO     900000
IDADE                  31
PROVINCIA         Malanje
PAG_1                  -1
INCUMPRIMENTO           0
Name: 927dd27f-1a76, dtype: object
```

Repare no `dtype: object` em baixo: a linha mistura números e texto, por isso a `Series` que a representa é do tipo `object`.

In [ ]:
# TAREFA: com iloc, mostre as três primeiras linhas e as colunas nas posições 0, 1, 5 e 6 de df
df.iloc[0:3, [0, 1, 5, 6]]

*Deverá obter:*

```
              ID  LIMITE_CREDITO  IDADE PROVINCIA
0  927dd27f-1a76          900000     31   Malanje
1  bd01a775-cec7               0      0         0
2  8d872d53-5776          350000     28    Luanda
```

**Pergunta 4.2.** A conta da Tarefa anterior é a primeira linha. E a segunda linha, `bd01a775-cec7`, faz sentido? Já a conhece de onde?

*Resposta:* A segunda linha corresponde à conta bd01a775-cec7 e limite de crédito é zero. Não é considerada uma conta válida segundo o critério da Tarefa 4.1 (LIMITE_CREDITO > 0).  A indentificamos através da máscara booleana (ok), que permite distinguir as contas válidas das restantes.

**Passo 4.3** — `groupby` no histórico de pagamentos: a taxa de incumprimento para cada código de `PAG_2` (Agosto).

In [ ]:
# TAREFA: só nas contas válidas (df[ok]), agrupe por PAG_2 e calcule contas (count de ID) e taxa
# (mean de INCUMPRIMENTO); arredonde a 3 casas
df[ok]
df[ok].groupby("PAG_2")
resultado = df[ok].groupby("PAG_2").agg(contas=("ID", "count"), taxa=("INCUMPRIMENTO", "mean")).round(3)
#print(resultado)

*Deverá obter:*

```
       contas   taxa
PAG_2
-2       3391  0.150
-1      10361  0.167
 0      12738  0.260
 1       2193  0.371
 2        610  0.559
 3        205  0.605
 4        109  0.679
 5         78  0.756
 6          9  0.778
 7          6  0.667
```

**Pergunta 4.3.** (a) Descreva o padrão da taxa à medida que o código aumenta. (b) Que códigos aparecem que **não** estão no dicionário de dados? Quantas contas têm? (c) Porque é que a taxa do código 6 (77,8 %) merece pouca confiança?

*Resposta:* (a) Descreva o padrão da taxa à medida que o código aumenta.                                                          

De modo geral, observamos uma tendência de aumento da taxa de incumprimento à medida que o código de PAG_2 aumenta. A taxa passa de 15% no código -2 para 77,8% no código 6. Contudo, essa tendência não é perfeitamente crescente, pois no código 7 a taxa diminui para 66,7%.

(b) Os códigos -2 e 0 não estão definidos no dicionário de dados. O código -2 aparece em 3 391 contas e o código 0 aparece em 12 738 contas.

(c) Porque é que a taxa do código 6 (77,8%) merece pouca confiança?

Porque esse grupo contém apenas 9 contas. A taxa de 77,8% corresponde a aproximadamente 7 contas em incumprimento. Uma única conta adicional ou retirada do grupo alteraria a taxa em cerca de 11,1 pontos percentuais. Portanto, a amostra é demasiado pequena para considerar essa percentagem uma estimativa estável.

*Lembre-se disto na Aula 7: ainda vamos verificar a qualidade de `PAG_2` a `PAG_6`.*

**Passo 4.4** — Os clientes jovens falham mais?

In [ ]:
# TAREFA: crie a máscara jovem = IDADE < 25. Nas contas válidas, calcule a taxa de incumprimento e o número
# de contas dos jovens e dos restantes, com groupby sobre a máscara (dica: df[ok].groupby(jovem[ok]))
jovem = df["IDADE"] < 25
resultado = df[ok].groupby(jovem[ok]).agg(contas=("ID", "count"), taxa=("INCUMPRIMENTO", "mean")).round(3)
print(resultado)


*Deverá obter:*

```
       count   mean
IDADE
False  29154  0.235
True     546  0.244
```

Um `groupby` também aceita uma máscara: os grupos são `False` (25 anos ou mais) e `True` (menos de 25). O índice chama-se `IDADE` porque a máscara foi criada a partir dessa coluna.

**Pergunta 4.4.** Os jovens falham mais? Use os dois números de cada grupo na resposta.

*Resposta:* ...

## Mensagem à directora de risco

Escreva 5 a 8 linhas, sempre com números: (1) como evoluiu o câmbio de compra de Abril a Setembro e qual é a margem média da tesouraria; (2) o que diz o código de pagamento de Agosto sobre o incumprimento, e que códigos não estão no dicionário; (3) se os clientes com menos de 25 anos falham mais.

*Mensagem:* ...

## 2.ª hora — Projecto do grupo

Abra o notebook do grupo e escreva a **§2 *Primeiro contacto***, seguindo a ficha da PL3 (Parte B): as cinco perguntas do primeiro contacto (Aula 1.9.3) aplicadas ao vosso conjunto de dados, com o código e as saídas, e o rascunho da **ficha do projecto**.

## Antes de entregar

1. **Kernel → Reiniciar e executar tudo** (no Colab: **Ambiente de execução → Reiniciar sessão e executar tudo**).
2. Confirme que todas as células correm sem erros e que as saídas coincidem com as indicadas (excepto as da Tarefa 1, que mudam com o dia).
3. Guarde como `PL03_<numero>.ipynb`.